# 05 - Diagnostics, Failure Audit, and Exports
This notebook performs a deep dive into the model's performance distribution using the saved diagnostic state. It visualizes the IoU spread, identifies localization champions, and systematically categorizes failures into distinct modes ('The Arrogant', 'The Clumsy', 'The Scattered') for further debugging. Finally, it exports these grouped insights as high-resolution visual assets.

In [ ]:
# State Restoration[cite: 1]
df_diag = pd.read_pickle('df_diag.pkl')
img_dir = Path(CFG["test_dir"]) / 'images/before'
mask_dir = Path(CFG["test_dir"]) / 'masks/before'
print(f"Loaded {len(df_diag)} diagnostic records.")

In [ ]:
# Cell 8.8: Deep IoU Analysis[cite: 1]
plt.figure(figsize=(12, 5))
sns.histplot(df_diag['IoU'], bins=20, kde=True, color='#4C72B0', alpha=0.7)
plt.axvline(df_diag['IoU'].mean(), color='red', linestyle='--', label=f'Mean IoU: {df_diag["IoU"].mean():.4f}')
plt.title("Distribution of Localization Accuracy (IoU) on Test Set", fontsize=14, fontweight='bold')
plt.xlabel("Intersection over Union (IoU) Score")
plt.ylabel("Number of Images")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

best_samples = df_diag[df_diag['Confidence'] > 0.5].nlargest(3, 'IoU')
worst_samples = df_diag[df_diag['Confidence'] > 0.5].nsmallest(3, 'IoU')

def plot_iou_gallery(samples_df, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for idx, row in enumerate(samples_df.itertuples()):
        path = Path(row.Filepath)
        img_pil = Image.open(path).convert('RGB')
        input_tensor = eval_transforms(img_pil).unsqueeze(0).to(CFG["device"])
        
        h_plus = grad_cam_plus_engine(input_tensor=input_tensor, targets=[ClassifierOutputTarget(BEFORE_IDX)])[0, :]
        h_plus = cv2.GaussianBlur(h_plus, CFG["smooth_kernel"], 0)
        h_plus = (h_plus - h_plus.min()) / (h_plus.max() - h_plus.min() + 1e-8)
        
        mask = get_final_mask(h_plus, model_confidence=row.Confidence)
        
        img_np = np.array(img_pil.resize((CFG["img_size"], CFG["img_size"])))
        overlay = img_np.copy()
        
        bool_mask = mask > 0
        overlay[bool_mask, 0] = 255 
        overlay[bool_mask, 1] = 0   
        overlay[bool_mask, 2] = 0   
        
        blended = np.where(bool_mask[:, :, None], cv2.addWeighted(img_np, 0.5, overlay, 0.5, 0), img_np)
        
        axes[idx].imshow(blended)
        axes[idx].set_title(f"IoU: {row.IoU:.4f}\nConf: {row.Confidence:.2f}", fontweight='bold')
        axes[idx].axis('off')
        
    plt.suptitle(title, fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

plot_iou_gallery(best_samples, "Localization Champions (Highest IoU)")
plot_iou_gallery(worst_samples, "Localization Failures (Lowest IoU)")